<a href="https://colab.research.google.com/github/Aadit-5810228/Aadit-5810228-AnalyticsEngineering/blob/main/5_02_practice_weekly_ops.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 5.02 Analytics Engineering: practice test
## Practice: Weekly operations pipeline

## How this practice works

This is a rehearsal for the in-class test. It is **not marked** and nothing is submitted. It works only if you treat it like the real thing: attempt it first, and only then look at the model answer.

This is set to be the same time given in the real test; the same style of notebook and problem; and as close to the real thing as possible. However, of course, the problem is different; the data is different; and you will do it asychronously (which is also different).

- Set a timer for **55 minutes** and work alone. Stop when it rings, then spend **5 minutes** exporting your notebook (see below). Write your start and stop times in the cell under the timing table.
- Work in **Google Colab** and save the notebook as `.ipynb`.
- You may use *any materials, the module GitHub code and any generative AI tool*.
- There is no end-point as such. The task is longer than 55 minutes, so you are completing part of a larger job. Do as much as you can, and do it properly.
- The code itself is not marked. Marks in the real test go on how the task was executed, the methodology applied, and the interpretation of results. Practise that.
- In text cells or comments, include:
  1. **how the code was generated**: the prompt you sent, or the source you used;
  2. **evidence of checking**: what you compared, counted or inspected, and what it showed;
  3. **your interpretation in your own words**.

**Tips**

- Run small pieces and know what output you expect before you run them.
- Put enough text next to the code that someone else can follow your reasoning.
- Follow sound analytics engineering method: understand the grain/rows that you are building around, clean in stages, check before you move on.
- Quality over quantity. A smaller pipeline that you have checked beats a larger one you have not.
- Stay calm. Not finishing is expected.

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Explore and profile the tables |
| 10 to 20 | Plan: grain, joins, cleaning decisions |
| 20 to 45 | Build as far as you can |
| 45 to 55 | Check, reconcile and write up |
| 55 to 60 | Export: **Runtime, then Restart session and run all**; File, Download, .ipynb; put the file in a zip |

In the real test you upload the zip to the Class Test area on my.wbs. Here, keep it so you can compare your work with the model answer afterwards.



```
# This is formatted as code
```

**Started at:** ____  **Stopped at:** ____  **Furthest step I reached (see the brief):** ____

## The data

You have six tables from a fictional food-delivery marketplace, Plateful, which works with restaurants in five UK cities. They cover orders placed from 1 March to 30 June 2026.

| Name in this notebook | What it holds |
| --- | --- |
| `orders` | One record per order: customer, restaurant, when it was placed, delivery fee |
| `order_status_events` | The order's progress through the process, recorded as events |
| `order_lines` | The items inside each order |
| `payments` | Money movements for each order: charges and refunds |
| `restaurants` | Restaurant details: cuisine and city |
| `customers` | Customer details |

All timestamps in all tables are stored in **UTC**.

**How the tables connect**

- `orders` to `order_status_events`, `order_lines` and `payments` on `order_id`
- `orders` to `restaurants` on `restaurant_id`
- `orders` to `customers` on `customer_id`

Check the actual column names in the next cell before you write any joins.

In [1]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/practicesession/plateful-practice.zip"    # link to plateful-practice.zip (the CSV files are inside a folder in the zip)

DATA_DIR = "/content/plateful"

if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/plateful.zip")
    zipfile.ZipFile("/content/plateful.zip").extractall(DATA_DIR)

FILES = {
    "orders": "orders.csv",
    "order_status_events": "order_status_events.csv",
    "order_lines": "order_lines.csv",
    "payments": "payments.csv",
    "restaurants": "restaurants.csv",
    "customers": "customers.csv",
}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}.")

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))

orders: 14,494 rows, 5 columns
    ['order_id', 'customer_id', 'restaurant_id', 'order_ts', 'delivery_fee']
order_status_events: 56,812 rows, 4 columns
    ['event_id', 'order_id', 'status', 'event_ts']
order_lines: 36,120 rows, 5 columns
    ['order_id', 'line_no', 'item_name', 'quantity', 'unit_price']
payments: 16,423 rows, 5 columns
    ['payment_id', 'order_id', 'payment_type', 'amount', 'paid_at']
restaurants: 67 rows, 5 columns
    ['restaurant_id', 'restaurant_name', 'cuisine', 'city', 'effective_from']
customers: 4,200 rows, 3 columns
    ['customer_id', 'signup_date', 'home_city']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each, using only the tables and keys described above. Run them and change them to see what they do. You do not have to use them in your answer.

In [ ]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

# Look at the first few rows of a table
display(q("SELECT * FROM orders LIMIT 5"))

# How many rows are there, and how many different order_id values?
display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_distinct_orders FROM orders"))

# Join two tables on a key
display(q("""
    SELECT o.order_id, r.restaurant_id
    FROM orders AS o
    JOIN restaurants AS r ON o.restaurant_id = r.restaurant_id
    LIMIT 5
"""))

In [ ]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

orders = tables["orders"]
display(orders.head())        # first few rows
print(orders.shape)           # (number of rows, number of columns)
print(orders.dtypes)          # data type of each column

# The same join as in Example 1, using pandas
joined = tables["orders"].merge(tables["restaurants"], on="restaurant_id", how="inner")
display(joined[["order_id", "restaurant_id"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT customer_id FROM orders")
print(result["customer_id"].nunique(), "different customer_id values in orders")

## Your brief

**From:** Operations

> We run Plateful, a food-delivery marketplace in five UK cities. We are starting a weekly operations report and need the first layers of a data pipeline behind it: clean tables first, then one reporting table. For each week, show orders, net revenue and average delivery time by cuisine and by city. We also need to know how far we can trust the figures.

**What your final table should look like**

One table with columns for week, cuisine, city, number of orders, net revenue and average delivery time in minutes. State clearly what one row of your table represents.

**Decisions you must make and state**

- Which week does an order belong to?
- Which orders count as orders, and which do not?
- What does "net revenue" mean here, and where does it come from?
- What counts as "delivery time", and for which orders can it be measured?

**A suggested route, in order** (you are not expected to finish)

1. Profile the tables: grain, keys, duplicates, odd values.
2. Build clean tables (a silver layer) with one row per order where that is the grain.
3. Build the one reporting table (gold) from them.
4. Show with checks that the gold table agrees with the source data.
5. If you have time: add cancelled orders and a refund rate to the table, and write automated checks for your clean tables.

## 1. Understand the data

*Look before you build. Write down what you find: what one row of each table represents, how the tables connect, and anything that looks odd or missing.*

First I want to have a glance over each of the 6 data sets. I will be doing this using head, information and description function. This is done to spot any major error and get a jist on how each table looks

In [2]:
#Promop: Give me table information, head, and decsription for the customers data
tables["customers"].info()
tables["customers"].head()
tables["customers"].describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4200 entries, 0 to 4199
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   customer_id  4200 non-null   object
 1   signup_date  4200 non-null   object
 2   home_city    4200 non-null   object
dtypes: object(3)
memory usage: 98.6+ KB


,customer_id,signup_date,home_city
count,4200,4200,4200
unique,4200,794,5
top,C04200,2025-09-13,London
freq,1,11,1541


This suggests that we have all unique customer IDs. Next we need to check for consistents formatting for signup dates and city names

In [3]:
#Prompt: Do the same analysis as above for the order_lines data
tables["order_lines"].info()
tables["order_lines"].head()
tables["order_lines"].describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36120 entries, 0 to 36119
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   order_id    36120 non-null  object 
 1   line_no     36120 non-null  int64  
 2   item_name   36120 non-null  object 
 3   quantity    36120 non-null  int64  
 4   unit_price  36120 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 1.4+ MB


,line_no,quantity,unit_price
count,36120.000000,36120.000000,36120.000000
mean,1.996152,1.995903,7.211739
std,0.998690,0.813985,3.131357
min,1.000000,1.000000,2.000000
25%,1.000000,1.000000,5.000000
50%,2.000000,2.000000,8.000000
75%,3.000000,3.000000,10.500000
max,4.000000,3.000000,12.000000


In [5]:
#Prompt: Do the same analysis as above for the order_status_events data
tables["order_status_events"].info()
tables["order_status_events"].head()
tables["order_status_events"].describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56812 entries, 0 to 56811
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   event_id  56812 non-null  object
 1   order_id  56812 non-null  object
 2   status    56812 non-null  object
 3   event_ts  56812 non-null  object
dtypes: object(4)
memory usage: 1.7+ MB


,event_id,order_id,status,event_ts
count,56812,56812,56812,56812
unique,56812,14494,5,56200
top,E020829,O003952,placed,2026-05-17T18:01:06Z
freq,1,6,14494,3


here the output suggests that we have multiple event ID for the same orders

In [7]:
#Prompt: Do the same analysis as above for the orders data
tables["orders"].info()
tables["orders"].head()
tables["orders"].describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14494 entries, 0 to 14493
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       14494 non-null  object 
 1   customer_id    14494 non-null  object 
 2   restaurant_id  14494 non-null  object 
 3   order_ts       14494 non-null  object 
 4   delivery_fee   14494 non-null  float64
dtypes: float64(1), object(4)
memory usage: 566.3+ KB


,delivery_fee
count,14494.000000
mean,2.150676
std,1.104824
min,0.000000
25%,1.490000
50%,2.490000
75%,2.990000
max,3.490000


In [9]:
#Prompt: Do the same analysis as above for the payments data
tables["payments"].info()
tables["payments"].head()
tables["payments"].describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16423 entries, 0 to 16422
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   payment_id    16423 non-null  object 
 1   order_id      16423 non-null  object 
 2   payment_type  16423 non-null  object 
 3   amount        16423 non-null  float64
 4   paid_at       16423 non-null  object 
dtypes: float64(1), object(4)
memory usage: 641.7+ KB


,amount
count,16423.000000
mean,30.988618
std,29.080373
min,-127.490000
25%,14.490000
50%,31.490000
75%,49.990000
max,127.490000


The negative value as the minimum value in the amounts column suggests that refunds initiated from the restaurant are also included in the table. Hence those orders need to be excluded from the cleaned data

In [10]:
#Prompt: Do the same analysis as above for the restaurants data
tables["restaurants"].info()
tables["restaurants"].head()
tables["restaurants"].describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67 entries, 0 to 66
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   restaurant_id    67 non-null     object
 1   restaurant_name  67 non-null     object
 2   cuisine          67 non-null     object
 3   city             67 non-null     object
 4   effective_from   67 non-null     object
dtypes: object(5)
memory usage: 2.7+ KB


,restaurant_id,restaurant_name,cuisine,city,effective_from
count,67,67,67,67,67
unique,64,64,15,8,4
top,R007,Royal Japanese 7,Vegan,London,2025-01-01
freq,2,2,9,23,64


The number of cities in this output = 8. Considering the restaurant only operates in 5 cities, this points to a formatting error. Also we have duplicates within restaturant ID column which also need to fixed.

## 2. Plan

The final table will consist of data aggregated from all the 6 tables. The left join function will be used to join all the tables to the Orders data set. Primary keys like order_id, restaturant_id, and customer_id will be used to do the same. Considering a weekly aggregation is required in the final output, the payment dates from the payments data set will be used to aggregate the total amounts in a weekly manner. This aggregation will include city wise and cusine wise weekly revenue generation, number of orders and average delivery time

Add text here

## 3. Build

*Build as far as you can in the time. Short, re-runnable cells are better than one long cell.*

First we need to make checks if there are any NA values and the formatting across the cateogrical data is consistent


In [16]:
#Prompt: Check the customers data set for any NA values across each column. Also do a check for formatting consistency acorss the categorical data

print("Missing values per column in 'customers':")
print(tables["customers"].isna().sum())

print("\nUnique values in 'home_city':")
print(tables["customers"]["home_city"].unique())

print("\nValue counts for 'home_city':")
print(tables["customers"]["home_city"].value_counts())

Missing values per column in 'customers':
customer_id    0
signup_date    0
home_city      0
dtype: int64

Unique values in 'home_city':
['London' 'Leeds' 'Manchester' 'Bristol' 'Birmingham']

Value counts for 'home_city':
home_city
London        1541
Manchester     857
Birmingham     697
Bristol        569
Leeds          536
Name: count, dtype: int64


This suggests that there are no NA values that need to be cleaned up and the fomatting within the home_city column is cosistent. These same checks now need to be done for the other data sets as well

In [18]:
#Do similar check for the order_line data
print("Missing values per column in 'order_lines':")
print(tables["order_lines"].isna().sum())

print("\nUnique values in 'item_name':")
print(tables["order_lines"]["item_name"].unique())

Missing values per column in 'order_lines':
order_id      0
line_no       0
item_name     0
quantity      0
unit_price    0
dtype: int64

Unique values in 'item_name':
['Ramen' 'Soft drink' 'Side salad' 'Burrito' 'Fries' 'Fried rice'
 'Chicken tikka' 'Pad thai' 'Dessert' 'Falafel wrap' 'Cheeseburger'
 'Margherita']


No NA values or incosistent formatting in the order_line data

In [23]:
#For the order_status_events data check the NA values and check if the formatting of the dates within the for the event_ts is consistent

# 1. Check for NA values
print("Missing values in order_status_events:")
print(tables['order_status_events'].isna().sum())

# 2. Check datetime format consistency for event_ts
# We will attempt to parse all event_ts values with standard UTC format validation
try:
    parsed_ts = pd.to_datetime(tables['order_status_events']['event_ts'], format='%Y-%m-%dT%H:%M:%SZ', errors='raise')
    print("\nSuccess: All event_ts values are formatted consistently as '%Y-%m-%dT%H:%M:%SZ'")
    print(f"Date range: {parsed_ts.min()} to {parsed_ts.max()}")
except Exception as e:
    print(f"\nFormatting inconsistency found: {e}")

Missing values in order_status_events:
event_id    0
order_id    0
status      0
event_ts    0
dtype: int64

Success: All event_ts values are formatted consistently as '%Y-%m-%dT%H:%M:%SZ'
Date range: 2026-03-01 09:13:23 to 2026-06-30 23:37:01


No NA values and consistent formatting of the dates in the order_status_events data

In [24]:
#Check the orders data set for any NA values and check is the data foramtting in the order_ts column is consistent
print("Missing values in orders:")
print(tables['orders'].isna().sum())

# Check datetime format consistency for order_ts
try:
    parsed_order_ts = pd.to_datetime(tables['orders']['order_ts'], format='%Y-%m-%dT%H:%M:%SZ', errors='raise')
    print("\nSuccess: All order_ts values are formatted consistently as '%Y-%m-%dT%H:%M:%SZ'")
    print(f"Date range: {parsed_order_ts.min()} to {parsed_order_ts.max()}")
except Exception as e:
    print(f"\nFormatting inconsistency found: {e}")

Missing values in orders:
order_id         0
customer_id      0
restaurant_id    0
order_ts         0
delivery_fee     0
dtype: int64

Success: All order_ts values are formatted consistently as '%Y-%m-%dT%H:%M:%SZ'
Date range: 2026-03-01 09:13:23 to 2026-06-30 22:58:46


No NA values and consistent formatting of the dates in the orders data

## 4. Check

*Show how you know your answer is right. What did you compare, count or spot check, and what did the checks show? If a check failed, say what you changed.*

Add text here as you go along. Show that you are staying in control of the build!

## 5. Prompts and AI notes

*Paste each prompt you used, as you sent it. Then note what you accepted, what you changed or rejected, and why. Include at least one example where the AI was wrong or needed correcting, if that happened.*

Add prompts here.

## 6. What I could not verify

*Be honest about what remains unchecked, and what you would do next with more time.*

Add text here.

## Self-check before you stop

Tick these off in your head or in a text cell. They mirror the three things the real test is marked on.

**How was the code generated?**
- [ ] Every block of code that matters has a prompt or a source associated with it.
- [ ] I can say what each block is meant to do and what output I expected.

**Evidence of checking**
- [ ] For every table I built, I checked the number of rows against what I expected, and I can say why.
- [ ] I checked that keys I treat as unique are unique, and I looked for duplicates.
- [ ] My final table agrees with the source data in at least one total, and I show the comparison.
- [ ] I looked at individual records, not only at totals.
- [ ] I wrote down at least one thing the checks caught, and what I changed (if possible).

**Interpretation in my own words**
- [ ] I state each decision (week, which orders count, net revenue, delivery time) and why I chose it.
- [ ] I say what the figures can and cannot be trusted for.
- [ ] Section 6 lists what I did not verify, honestly.